# BPR — régimen **F2** sobre **KOZYRIEV (+FronkonGames)**

Full-ranking · split random 80/10/10 · métricas `Recall/NDCG/Hit/Precision@5,10` + `Cov/Ent` + long-tail · **1 seed** (Colab gratuito).

> Carga via **kagglehub**. Positivo = `is_recommended`; `playtime`=horas. k-core(5,20).

> Plantilla = `deep_kozyriev_h3.ipynb`; modelo enchufado al harness F2 importado de `recsys_protocol.py`. Las filas MostPop/ALS son **self-check** (deben reproducir las del `*_corregido`).

In [1]:
# ====== Bootstrap: deps + clonar repo + cargar recsys_protocol ======
!pip install -q kagglehub implicit
import os, sys, glob, json, math, time, csv, zipfile, shutil, subprocess, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

REPO_URL = 'https://github.com/Benjaa7/Proyecto-RecSys.git'
REPO_DIR = '/content/Proyecto-RecSys'
def _locate_protocol():
    if os.path.isdir(os.path.join(REPO_DIR, '.git')):
        subprocess.run(['git','-C',REPO_DIR,'fetch','-q','--depth','1','origin'], check=False)
        subprocess.run(['git','-C',REPO_DIR,'reset','--hard','-q','FETCH_HEAD'], check=False)
    else:
        subprocess.run(['git','clone','--depth','1','-q',REPO_URL,REPO_DIR], check=False)
    h3 = os.path.join(REPO_DIR, 'H3')
    if os.path.exists(os.path.join(h3, 'recsys_protocol.py')):
        return h3
    for c in ['/content','.','..','H3','../H3'] + sorted(glob.glob('/content/drive/MyDrive/*')):
        if c and os.path.exists(os.path.join(c, 'recsys_protocol.py')):
            return c
    return None
_p = _locate_protocol()
assert _p, 'No encontre recsys_protocol.py (clona el repo o sube el modulo).'
if _p not in sys.path: sys.path.insert(0, _p)

from recsys_protocol import SEED, set_global_seed, iterative_k_core
set_global_seed()
# Sección F2: usar la del módulo si está publicada; si el repo clonado trae una
# versión vieja de recsys_protocol (sin F2), caer a definiciones inline IDÉNTICAS.
try:
    from recsys_protocol import random_split_8010, paper_metrics, cat_cov_ent, recs_from_embeddings, longtail_ndcg
    print('[F2] funciones importadas de recsys_protocol')
except ImportError:
    print('[F2] recsys_protocol clonado sin sección F2 -> usando definiciones inline (idénticas al módulo)')
    def random_split_8010(inter, seed=SEED):
        inter = inter.reset_index(drop=True); rng = np.random.default_rng(seed); n = len(inter)
        inter = inter.iloc[rng.permutation(n)].reset_index(drop=True)
        n_tr, n_va = int(0.8 * n), int(0.1 * n)
        sp = np.empty(n, dtype='int8'); sp[:n_tr] = 0; sp[n_tr:n_tr + n_va] = 1; sp[n_tr + n_va:] = 2
        inter['split'] = sp; return inter
    def _dcg_f2(hits): return sum((1.0 / math.log2(i + 2)) for i, h in enumerate(hits) if h)
    def cat_cov_ent(recs, cat_map, k):
        covs, ents = [], []
        for rec in recs.values():
            cnt = {}
            for it in rec[:k]:
                for c in cat_map.get(it, ()): cnt[c] = cnt.get(c, 0) + 1
            if not cnt: covs.append(0); ents.append(0.0); continue
            covs.append(len(cnt)); tot = sum(cnt.values())
            ents.append(-sum((v / tot) * math.log2(v / tot) for v in cnt.values()))
        return (float(np.mean(covs)) if covs else 0.0, float(np.mean(ents)) if ents else 0.0)
    def paper_metrics(recs, test_items, ks=(5, 10), cat_maps=None):
        acc = {f'{m}@{k}': [] for k in ks for m in ('Recall', 'NDCG', 'Hit', 'Precision')}; n = 0
        for u, rec in recs.items():
            rel = test_items.get(u)
            if not rel: continue
            n += 1
            for k in ks:
                hits = [(1 if it in rel else 0) for it in rec[:k]]; nhit = sum(hits)
                acc[f'Recall@{k}'].append(nhit / len(rel)); acc[f'Precision@{k}'].append(nhit / k)
                acc[f'Hit@{k}'].append(1.0 if nhit > 0 else 0.0)
                idcg = sum(1.0 / math.log2(i + 2) for i in range(min(len(rel), k)))
                acc[f'NDCG@{k}'].append(_dcg_f2(hits) / idcg if idcg > 0 else 0.0)
        out = {key: (float(np.mean(v)) if v else 0.0) for key, v in acc.items()}
        if cat_maps:
            for k in ks:
                for name, cmap in cat_maps.items():
                    cov, ent = cat_cov_ent(recs, cmap, k); out[f'Cov_{name}@{k}'] = cov; out[f'Ent_{name}@{k}'] = ent
        out['n_users'] = n; return out
    def recs_from_embeddings(e_u, e_i, umap, idx2app, users, topn, train_items_per_user, popular_list):
        out = {}; nfb = 0
        for u in users:
            seen = train_items_per_user.get(u, set()); key = str(u)
            if key not in umap:
                out[u] = [i for i in popular_list if i not in seen][:topn]; nfb += 1; continue
            scores = e_i @ e_u[umap[key]]; rec = []
            for j in np.argsort(-scores):
                a = idx2app[int(j)]
                if a not in seen:
                    rec.append(a)
                    if len(rec) >= topn: break
            out[u] = rec
        return out, nfb
    def _bucket_f2(n): return '2-5' if n <= 5 else '6-20' if n <= 20 else '21-50' if n <= 50 else '51+'
    def longtail_ndcg(recs, test_items, train_items, k=10, buckets=('2-5', '6-20', '21-50', '51+')):
        def _u_nr(rec, rel, kk):
            hits = [1 if it in rel else 0 for it in rec[:kk]]; nh = sum(hits)
            dcg = sum(1 / math.log2(i + 2) for i, h in enumerate(hits) if h)
            idcg = sum(1 / math.log2(i + 2) for i in range(min(len(rel), kk)))
            return (dcg / idcg if idcg > 0 else 0.0, nh / len(rel) if rel else 0.0)
        act = {u: _bucket_f2(len(train_items.get(u, set()))) for u in recs}; out = {}
        for b in buckets:
            us = [u for u in recs if act.get(u) == b and test_items.get(u)]
            if not us: continue
            out[b] = {'n': len(us),
                      f'NDCG@{k}': float(np.mean([_u_nr(recs[u], test_items[u], k)[0] for u in us])),
                      f'Recall@{k}': float(np.mean([_u_nr(recs[u], test_items[u], k)[1] for u in us]))}
        return out
print('bootstrap OK | numpy', np.__version__, '| pandas', pd.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 49.0 MB/s eta 0:00:00
[protocol] v2026-06-23b (defaults: eval sin tope max_eval_users=None + frac_train=0.30)
[protocol] seed global = 42 | numpy/random/torch (cuda=True, determinista=True)
[F2] recsys_protocol clonado sin sección F2 -> usando definiciones inline (idénticas al módulo)
bootstrap OK | numpy 2.0.2 | pandas 2.2.2


In [2]:
# ====== Config — KOZYRIEV (k-core 5/20, split random 80/10/10) ======
DATASET = 'kozyriev'
MODEL_NAME = 'BPR'
MIN_USER, MIN_GAME = 5, 20
KS = (5, 10)
ALS_FACTORS, ALS_ITERS, ALS_REG, ALS_ALPHA = 64, 15, 0.1, 40.0
N_EVAL_ANALYSIS = 200_000                 # mismo eval_set objetivo que el *_corregido
TIER = 'T2'                               # 'T1' = sanity (eval cap 2000) | 'T2' = final
SUBSAMPLE_USERS = 500_000 if TIER == 'T2' else 200_000
N_EXAMPLES = 3
OUT = f'bpr_kozyriev_h3'; os.makedirs(OUT, exist_ok=True)
print(f'BPR | KOZYRIEV | TIER={TIER} | k-core({MIN_USER},{MIN_GAME}) | '
      f'subsample={SUBSAMPLE_USERS} | N_EVAL={N_EVAL_ANALYSIS}')


BPR | KOZYRIEV | TIER=T2 | k-core(5,20) | subsample=500000 | N_EVAL=200000


In [3]:
# ====== Carga KOZYRIEV: kagglehub + k-core(5,20) + positivos + split 80/10/10 ======
import kagglehub
SLUG, FRONKON_SLUG = 'antonkozyriev/game-recommendations-on-steam', 'fronkongames/steam-games-dataset'
def _find_file(root, *names):
    cand = {n.lower() for n in names}
    for p in Path(root).rglob('*'):
        if p.is_file() and p.name.lower() in cand: return p
    raise FileNotFoundError(f'No encontre {names} bajo {root}')
def _maybe_unzip(p, *wanted):
    with open(p,'rb') as f:
        if f.read(4) != b'PK\x03\x04': return p
    ext = p.parent/(p.stem+'_x'); ext.mkdir(exist_ok=True)
    with zipfile.ZipFile(p) as z:
        names=z.namelist(); target=None
        for w in wanted:
            target=next((n for n in names if Path(n).name.lower()==w.lower()), None)
            if target: break
        target=target or next((n for n in names if n.lower().endswith(('.csv','.json'))), names[0])
        out=ext/Path(target).name
        if not out.exists() or out.stat().st_size==0:
            with z.open(target) as s, open(out,'wb') as d: shutil.copyfileobj(s,d)
        return out
def fetch_file(slug, *fns):
    for fn in fns:
        try:
            p=Path(kagglehub.dataset_download(slug, path=fn)); p=p if p.is_file() else _find_file(p,fn)
            return _maybe_unzip(p,*fns)
        except Exception: continue
    return _find_file(Path(kagglehub.dataset_download(slug)), *fns)
def read_csv_robust(path, **kw):
    for enc in ('utf-8','utf-8-sig','cp1252','latin-1'):
        try: return pd.read_csv(path, encoding=enc, **kw)
        except (UnicodeDecodeError, UnicodeError): continue
    return pd.read_csv(path, encoding='latin-1', encoding_errors='replace', **kw)

t0=time.time()
games_csv=fetch_file(SLUG,'games.csv'); meta_json=fetch_file(SLUG,'games_metadata.json')
recs_csv=fetch_file(SLUG,'recommendations.csv'); fg_csv=fetch_file(FRONKON_SLUG,'games.csv')
print(f'descargado en {time.time()-t0:.0f}s')

parts=[]
for chunk in pd.read_csv(recs_csv, usecols=['app_id','user_id','hours','is_recommended','date'],
                         chunksize=2_000_000,
                         dtype={'app_id':'int32','user_id':'int32','hours':'float32','is_recommended':'bool'}):
    parts.append(chunk)
recs_full=pd.concat(parts, ignore_index=True); del parts
recs_full=recs_full.drop_duplicates(['user_id','app_id'], keep='last', ignore_index=True)
recs_filtered=iterative_k_core(recs_full, MIN_USER, MIN_GAME, user_col='user_id', item_col='app_id', verbose=True)
del recs_full
print(f'k-core: {recs_filtered["app_id"].nunique():,} juegos | {len(recs_filtered):,} interac')

pos=recs_filtered[recs_filtered['is_recommended']][['user_id','app_id','hours']].copy()
pos.columns=['user_id','app_id','playtime']
pos['user_id']=pos['user_id'].astype('int64'); pos['app_id']=pos['app_id'].astype('int64')
if SUBSAMPLE_USERS is not None:
    _rng=np.random.default_rng(SEED); _u=np.sort(pos['user_id'].unique())
    keep=set(_rng.choice(_u, size=min(SUBSAMPLE_USERS,len(_u)), replace=False).tolist())
    pos=pos[pos['user_id'].isin(keep)].copy()
inter=random_split_8010(pos, SEED)
print(f'inter={len(inter):,} | usuarios={inter["user_id"].nunique():,} | juegos={inter["app_id"].nunique():,}')


Using Colab cache for faster access to the 'game-recommendations-on-steam' dataset.
Using Colab cache for faster access to the 'game-recommendations-on-steam' dataset.
Using Colab cache for faster access to the 'game-recommendations-on-steam' dataset.
Using Colab cache for faster access to the 'steam-games-dataset' dataset.
descargado en 11s
  iter 1: 41,154,773 → 22,339,679
  iter 2: 22,339,679 → 22,294,605
  iter 3: 22,294,605 → 22,287,486
  iter 4: 22,287,486 → 22,287,105
  iter 5: 22,287,105 → 22,287,029
  iter 6: 22,287,029 → 22,286,991
  iter 7: 22,286,991 → 22,286,979
  iter 8: 22,286,979 → 22,286,960
  iter 9: 22,286,960 → 22,286,960
k-core: 22,676 juegos | 22,286,960 interac
inter=4,976,211 | usuarios=500,000 | juegos=22,590


In [4]:
# ====== Categorias/contenido KOZYRIEV (FronkonGames + games.csv + metadata) ======
CATALOG=sorted(int(a) for a in inter['app_id'].unique()); catalog_set=set(CATALOG); n_catalog=len(CATALOG)
def _pick(df,*al):
    low={c.lower():c for c in df.columns}
    for a in al:
        if a.lower() in low: return low[a.lower()]
    return None
def _to_int64(s): return pd.to_numeric(s, errors='coerce').astype('Int64')
with open(fg_csv, encoding='utf-8', errors='replace', newline='') as f:
    header=next(csv.reader(f))
fixed,rep=[],False
for h in header:
    if h.strip().lower() in ('discountdlc count','discount dlc count'): fixed+=['Discount','DLC count']; rep=True
    else: fixed.append(h)
fgdf=(read_csv_robust(fg_csv, engine='python', on_bad_lines='skip', dtype=str, header=0, names=fixed) if rep
      else read_csv_robust(fg_csv, engine='python', on_bad_lines='skip', dtype=str, index_col=False))
fg=pd.DataFrame()
fg['app_id']=_to_int64(fgdf[_pick(fgdf,'AppID','app_id','appid','steam_appid')])
fg['genres']=fgdf[_pick(fgdf,'Genres','genres')]; fg['developers']=fgdf[_pick(fgdf,'Developers','developers')]
fg['publishers']=fgdf[_pick(fgdf,'Publishers','publishers')]
fg=fg.dropna(subset=['app_id']); fg['app_id']=fg['app_id'].astype('int64'); fg=fg.drop_duplicates('app_id').set_index('app_id')
gfull=read_csv_robust(games_csv); gfull['app_id']=pd.to_numeric(gfull['app_id'], errors='coerce')
c_title=_pick(gfull,'title','name'); c_rat=_pick(gfull,'positive_ratio','rating')
gfull=gfull.dropna(subset=['app_id']); gfull['app_id']=gfull['app_id'].astype('int64'); gfull=gfull.drop_duplicates('app_id').set_index('app_id')
desc_map,tags_map={},{}
with open(meta_json,'r',encoding='utf-8') as f:
    for line in f:
        line=line.strip()
        if not line: continue
        try: rec=json.loads(line)
        except json.JSONDecodeError: continue
        aid=rec.get('app_id')
        if aid is None: continue
        aid=int(aid); desc_map[aid]=(rec.get('description') or '').strip(); tags_map[aid]=rec.get('tags') or []
def _split_multi(x):
    if x is None or (isinstance(x,float) and pd.isna(x)): return []
    items=[str(t) for t in x] if isinstance(x,(list,tuple,np.ndarray)) else str(x).split(',')
    return [t.strip().replace(',',' ') for t in items if t and str(t).strip().lower()!='nan']
def _fg(a,c):
    try: return fg.at[a,c]
    except Exception: return None
def _gv(a,c):
    try: return gfull.at[a,c] if c else None
    except Exception: return None
rows=[]
for a in CATALOG:
    g=_split_multi(_fg(a,'genres'))
    if not g: g=_split_multi(tags_map.get(a))
    d=_split_multi(_fg(a,'developers')); p=_split_multi(_fg(a,'publishers'))
    rat=pd.to_numeric(_gv(a,c_rat), errors='coerce') if c_rat else np.nan
    nm=_gv(a,c_title) if c_title else None
    rows.append({'app_id':a,'name':(str(nm) if isinstance(nm,str) and nm.strip() else f'app_{a}'),
                 'metascore':(float(rat) if pd.notna(rat) else np.nan),
                 'genres':g,'developers':d,'publishers':p,'description':desc_map.get(a,'')})
cat=pd.DataFrame(rows)
cat['metascore']=cat['metascore'].fillna(float(pd.to_numeric(cat['metascore'],errors='coerce').median()))
print(f'cat={len(cat):,} | con genero={sum(1 for a in CATALOG if cat.set_index("app_id")["genres"].get(a))}')


cat=22,590 | con genero=22583


In [5]:
# ====== Mapas de categoria + train/test + por-usuario (comun) ======
genre_map={int(a):list(g) for a,g in zip(cat['app_id'],cat['genres'])}
dev_map  ={int(a):list(g) for a,g in zip(cat['app_id'],cat['developers'])}
pub_map  ={int(a):list(g) for a,g in zip(cat['app_id'],cat['publishers'])}
total_map={a:[('g',x) for x in genre_map.get(a,[])]+[('d',x) for x in dev_map.get(a,[])]
              +[('p',x) for x in pub_map.get(a,[])] for a in CATALOG}
CAT_MAPS={'gene':genre_map,'dev':dev_map,'pub':pub_map,'total':total_map}
name_map={int(a):n for a,n in zip(cat['app_id'],cat['name'])}

train=inter[inter['split']==0]; test=inter[inter['split']==2]
train_items_per_user=train.groupby('user_id')['app_id'].apply(lambda s:set(int(x) for x in s)).to_dict()
test_items_per_user ={u:set(int(x) for x in g) for u,g in test.groupby('user_id')['app_id']}
eval_users=[u for u in test_items_per_user if u in train_items_per_user]
print(f'train={len(train):,} test={len(test):,} | eval usuarios={len(eval_users):,} | '
      f'positivos/usuario(medio)={np.mean([len(test_items_per_user[u]) for u in eval_users]):.2f}')


train=3,980,968 test=497,622 | eval usuarios=272,651 | positivos/usuario(medio)=1.82


In [6]:
# ====== MostPop + ALS (self-check) + eval_set fijo ======
import scipy.sparse as _sp
from implicit.als import AlternatingLeastSquares
pop=train.groupby('app_id').size().to_dict()
popular_list=[it for it,_ in sorted(pop.items(), key=lambda kv:(-kv[1],kv[0])) if it in catalog_set]
TOPN=max(KS)

als_users=sorted(train['user_id'].unique().tolist())
_u2i={u:i for i,u in enumerate(als_users)}; _a2i={a:i for i,a in enumerate(CATALOG)}
idx2app_als={i:a for a,i in _a2i.items()}
_tr=train[train['app_id'].isin(_a2i)]
_rows=_tr['user_id'].map(_u2i).to_numpy(); _cols=_tr['app_id'].map(_a2i).to_numpy()
_conf=(1.0+ALS_ALPHA*np.log1p(_tr['playtime'].fillna(0).clip(lower=0).to_numpy())).astype('float32')
_ui=_sp.csr_matrix((_conf,(_rows,_cols)), shape=(len(als_users),len(CATALOG)))
als=AlternatingLeastSquares(factors=ALS_FACTORS, regularization=ALS_REG, iterations=ALS_ITERS, random_state=SEED, use_gpu=False)
als.fit(_ui)
als_uf=np.asarray(als.user_factors); als_if=np.asarray(als.item_factors)
als_umap={str(u):_u2i[u] for u in als_users}
print('ALS listo:', als_uf.shape, als_if.shape)

# eval_set fijo (mismo criterio que el *_corregido: seed 42, tope N_EVAL_ANALYSIS)
if TIER=='T1': eval_users=eval_users[:2000]
if N_EVAL_ANALYSIS and len(eval_users)>N_EVAL_ANALYSIS:
    _rng=np.random.default_rng(SEED)
    eval_set=sorted(_rng.choice(np.array(eval_users), size=N_EVAL_ANALYSIS, replace=False).tolist())
else:
    eval_set=list(eval_users)
print(f'eval_set: {len(eval_set):,} usuarios')

recs_fixed={}
recs_fixed['Most Popular']={u:[i for i in popular_list if i not in train_items_per_user.get(u,set())][:TOPN] for u in eval_set}
recs_fixed['ALS'],_=recs_from_embeddings(als_uf,als_if,als_umap,idx2app_als,eval_set,TOPN,train_items_per_user,popular_list)
metrics_fixed={m:paper_metrics(recs_fixed[m],test_items_per_user,ks=KS,cat_maps=CAT_MAPS) for m in recs_fixed}
for m in recs_fixed:
    d=metrics_fixed[m]
    print(f'  {m:13s} R@5={d["Recall@5"]:.4f} NDCG@5={d["NDCG@5"]:.4f} NDCG@10={d["NDCG@10"]:.4f} '
          f'Hit@10={d["Hit@10"]:.4f} (self-check vs *_corregido)')


  0%|          | 0/15 [00:00<?, ?it/s]

ALS listo: (498829, 64) (22590, 64)
eval_set: 200,000 usuarios
  Most Popular  R@5=0.0289 NDCG@5=0.0207 NDCG@10=0.0282 Hit@10=0.0797 (self-check vs *_corregido)
  ALS           R@5=0.0586 NDCG@5=0.0436 NDCG@10=0.0559 Hit@10=0.1452 (self-check vs *_corregido)


## Modelo: BPR

In [7]:
# ====== BPR (implicit) — matriz BINARIA, misma indexacion que ALS ======
try:
    from implicit.bpr import BayesianPersonalizedRanking
except ImportError:
    from implicit.cpu.bpr import BayesianPersonalizedRanking
_uib=_sp.csr_matrix((np.ones(len(_tr),dtype='float32'),(_rows,_cols)), shape=(len(als_users),len(CATALOG)))
bpr=BayesianPersonalizedRanking(factors=64, learning_rate=0.01, regularization=0.01, iterations=100, random_state=SEED)
bpr.fit(_uib)
bpr_uf=np.asarray(bpr.user_factors); bpr_if=np.asarray(bpr.item_factors)   # (N,F+1)/(M,F+1): bias plegado
recs_fixed['BPR'],_nfb=recs_from_embeddings(bpr_uf,bpr_if,als_umap,idx2app_als,eval_set,TOPN,train_items_per_user,popular_list)
metrics_fixed['BPR']=paper_metrics(recs_fixed['BPR'],test_items_per_user,ks=KS,cat_maps=CAT_MAPS)
d=metrics_fixed['BPR']
print(f'BPR  R@5={d["Recall@5"]:.4f} NDCG@5={d["NDCG@5"]:.4f} NDCG@10={d["NDCG@10"]:.4f} '
      f'Hit@10={d["Hit@10"]:.4f} | cold-fallback={_nfb}')


  0%|          | 0/100 [00:00<?, ?it/s]

BPR  R@5=0.0392 NDCG@5=0.0295 NDCG@10=0.0375 Hit@10=0.0978 | cold-fallback=0


## Análisis (A accuracy · B diversidad · C long-tail · D ejemplos)

In [8]:
# ====== A/B/C/D (accuracy, diversidad, long-tail, ejemplos) + guardar + imprimir ======
import json as _json
MODELS=['Most Popular','ALS',MODEL_NAME]
_MET=['Recall@5','NDCG@5','Hit@5','Precision@5','Recall@10','NDCG@10']
dfA=pd.DataFrame([[m]+[f'{metrics_fixed[m][x]:.4f}' for x in _MET] for m in MODELS], columns=['Modelo']+_MET)
print('=== A. Accuracy (split 80/10/10, full-ranking, 1 seed) ===')
print(dfA.to_string(index=False))

DIV=[f'Cov_total@{k}' for k in KS]+[f'Cov_gene@{k}' for k in KS]+[f'Ent_gene@{k}' for k in KS]
dfB=pd.DataFrame({m:{c:f'{metrics_fixed[m][c]:.4f}' for c in DIV} for m in MODELS}).T[DIV]
print('\n=== B. Diversidad (Cov=nro categorias distintas en top-K; Ent=entropia) ===')
print(dfB.to_string())

lt={m:longtail_ndcg(recs_fixed[m],test_items_per_user,train_items_per_user,k=10) for m in MODELS}
buckets=['2-5','6-20','21-50','51+']
rowsC=[]
for b in buckets:
    row={'actividad':b}
    for m in MODELS: row[m]=(f'{lt[m][b]["NDCG@10"]:.4f}' if b in lt[m] else '')
    rowsC.append(row)
dfC=pd.DataFrame(rowsC)[['actividad']+MODELS]
print('\n=== C. Long-tail: NDCG@10 por actividad del usuario ===')
print(dfC.to_string(index=False))
print('n usuarios/bucket:', {b:(lt['Most Popular'][b]['n'] if b in lt['Most Popular'] else 0) for b in buckets})

def _u_nr5(rec,rel):
    hits=[1 if it in rel else 0 for it in rec[:5]]; nh=sum(hits)
    dcg=sum(1/math.log2(i+2) for i,h in enumerate(hits) if h)
    idcg=sum(1/math.log2(i+2) for i in range(min(len(rel),5)))
    return (dcg/idcg if idcg>0 else 0.0, nh/len(rel) if rel else 0.0)
_act={u:('2-5' if len(train_items_per_user.get(u,set()))<=5 else '6-20' if len(train_items_per_user.get(u,set()))<=20
         else '21-50' if len(train_items_per_user.get(u,set()))<=50 else '51+') for u in eval_set}
ej=[f'(ejemplos seed {SEED})']; _chosen=[]
for b in buckets:
    for u in [x for x in eval_set if _act[x]==b and test_items_per_user.get(x)]:
        if all(u in recs_fixed[m] for m in MODELS) and any(any(it in test_items_per_user[u] for it in recs_fixed[m][u][:5]) for m in MODELS):
            _chosen.append((b,u)); break
    if len(_chosen)>=N_EXAMPLES: break
def _nm(items): return [name_map.get(a,str(a)) for a in items]
for b,u in _chosen[:N_EXAMPLES]:
    rel=test_items_per_user[u]; hist=sorted(train_items_per_user.get(u,set()))
    ej.append(f'\n### Usuario {u} (actividad {b}, {len(hist)} juegos en historial)')
    ej.append('- Perfil (muestra): '+', '.join(_nm(hist[:6])))
    ej.append('- Test (a acertar): '+', '.join(_nm(sorted(rel))))
    for m in MODELS:
        nd,rc=_u_nr5(recs_fixed[m][u],rel)
        marks=[name_map.get(a,str(a))+(' ✓' if a in rel else '') for a in recs_fixed[m][u][:5]]
        ej.append(f'  - **{m}** (R@5={rc:.2f}, NDCG@5={nd:.2f}): '+', '.join(marks))
ej_txt='\n'.join(ej)
print('\n=== D. Ejemplos ===\n'+ej_txt)

dfA.to_csv(f'{OUT}/accuracy.csv', index=False); dfB.to_csv(f'{OUT}/diversidad.csv'); dfC.to_csv(f'{OUT}/longtail.csv', index=False)
open(f'{OUT}/ejemplos.md','w',encoding='utf-8').write(ej_txt)
_full={'dataset':DATASET,'model':MODEL_NAME,'seed':SEED,'tier':TIER,'n_eval':len(eval_set),
       'metrics':{m:{k:(float(v) if isinstance(v,(int,float,np.floating)) else v) for k,v in metrics_fixed[m].items()} for m in MODELS}}
_json.dump(_full, open(f'{OUT}/metricas_full.json','w'), indent=2)
print('\n'+'#'*72+'\n# RESPALDO EN TEXTO (todo impreso por si no se descargan archivos)\n'+'#'*72)
print('\n== accuracy.csv ==\n'+dfA.to_csv(index=False))
print('== diversidad.csv ==\n'+dfB.to_csv())
print('== longtail.csv ==\n'+dfC.to_csv(index=False))
print('== metricas_full.json ==\n'+_json.dumps(_full, indent=2))
try:
    shutil.make_archive(OUT,'zip',OUT)
    from google.colab import files; files.download(OUT+'.zip'); print('(zip de descarga generado)')
except Exception as e:
    print('(descarga automatica no disponible:', repr(e), '-> todo esta impreso arriba)')


=== A. Accuracy (split 80/10/10, full-ranking, 1 seed) ===
      Modelo Recall@5 NDCG@5  Hit@5 Precision@5 Recall@10 NDCG@10
Most Popular   0.0289 0.0207 0.0466      0.0095    0.0500  0.0282
         ALS   0.0586 0.0436 0.0927      0.0194    0.0931  0.0559
         BPR   0.0392 0.0295 0.0627      0.0130    0.0618  0.0375

=== B. Diversidad (Cov=nro categorias distintas en top-K; Ent=entropia) ===
             Cov_total@5 Cov_total@10 Cov_gene@5 Cov_gene@10 Ent_gene@5 Ent_gene@10
Most Popular     13.2255      29.2414     4.0198     10.8365     1.7374      3.1607
ALS              18.4826      31.9133     6.7256      9.3655     2.3930      2.7240
BPR              19.2869      33.0164     8.7605     12.8370     2.5517      2.9242

=== C. Long-tail: NDCG@10 por actividad del usuario ===
actividad Most Popular    ALS    BPR
      2-5       0.0292 0.0534 0.0370
     6-20       0.0277 0.0585 0.0380
    21-50       0.0258 0.0551 0.0362
      51+       0.0244 0.0534 0.0446
n usuarios/bucket: {'2

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

(zip de descarga generado)
